# Lab 00 — Environment Setup Check

Run this notebook **before starting Lab 01**. It validates that every dependency
is installed and that RustFS is reachable. Each check prints ✅ or ❌.

If any check fails, follow the fix instructions printed alongside the error.

---
## 1 · Python Version

In [ ]:
import sys

required = (3, 12)
actual = sys.version_info[:2]

if actual >= required:
    print(f'✅ Python {sys.version.split()[0]} — OK (>= 3.12 required)')
else:
    print(f'❌ Python {sys.version.split()[0]} — TOO OLD')
    print('   Fix: install Python 3.12+ and re-run make setup-env')

---
## 2 · Required Packages

In [ ]:
import importlib
import importlib.metadata

REQUIRED = {
    'boto3':      '1.34',
    'pandas':     '2.2',
    'pyarrow':    '15.0',
    'jupyterlab': '4.1',
    'ipywidgets': '8.1',
    'matplotlib': '3.8',
}

all_ok = True
for pkg, min_ver in REQUIRED.items():
    try:
        ver = importlib.metadata.version(pkg)
        print(f'  ✅ {pkg:<14} {ver}')
    except importlib.metadata.PackageNotFoundError:
        print(f'  ❌ {pkg:<14} NOT INSTALLED — run: uv sync')
        all_ok = False

print()
if all_ok:
    print('✅ All packages installed')
else:
    print('❌ Missing packages — run: make setup-env')

---
## 3 · RustFS Server Connectivity

In [ ]:
import urllib.request
import urllib.error

HEALTH_URL = 'http://localhost:9000/minio/health/live'

try:
    urllib.request.urlopen(HEALTH_URL, timeout=3)
    print('✅ RustFS is reachable at http://localhost:9000')
    print('   Admin Console: http://localhost:9001  (admin / adminpassword)')
except urllib.error.URLError:
    print('❌ RustFS is NOT reachable')
    print('   Fix: run  make up  in your terminal, then retry this cell')

---
## 4 · Default Buckets (bronze / silver / gold)

In [ ]:
import sys
sys.path.insert(0, '../scripts')
from lab_utils import get_s3_client

EXPECTED_BUCKETS = {'bronze', 'silver', 'gold'}

try:
    s3 = get_s3_client()
    existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
    missing = EXPECTED_BUCKETS - existing
    extra = existing - EXPECTED_BUCKETS

    for bucket in sorted(EXPECTED_BUCKETS):
        mark = '✅' if bucket in existing else '❌'
        print(f'  {mark} {bucket}')

    if extra:
        print(f'  ℹ️  Extra buckets: {", ".join(sorted(extra))}')

    print()
    if not missing:
        print('✅ All default buckets present')
    else:
        print(f'❌ Missing buckets: {", ".join(sorted(missing))}')
        print('   Fix: run  make down && make up  to re-run the init container')
except Exception as e:
    print(f'❌ Could not connect to RustFS: {e}')
    print('   Fix: run  make up  in your terminal first')

---
## 5 · Lab Utilities Module

In [ ]:
import sys
sys.path.insert(0, '../scripts')

try:
    import lab_utils
    fns = [f for f in dir(lab_utils) if not f.startswith('_')]
    print(f'✅ lab_utils loaded — exports: {", ".join(fns)}')
except ImportError as e:
    print(f'❌ lab_utils not found: {e}')
    print('   Expected at: ../scripts/lab_utils.py')

---
## Summary

If all cells printed ✅, your environment is ready. Start with **Lab 01**.

| Lab | Topic |
|-----|-------|
| 01 | boto3 basics — upload, download, presigned URLs |
| 02 | Medallion architecture — Bronze → Silver → Gold with Parquet |
| 03 | Multipart upload — parallel parts, abort, integrity check |
| 04 | Fault tolerance — drive failure simulation with RS(4,2) |
| 05 | Object versioning — delete markers, restore, lifecycle rules |
| 06 | Erasure coding in practice — shard distribution, EC math |
| 07 | Reed-Solomon from scratch — implement the math in pure Python |
| 08 | Benchmarking — throughput comparison with matplotlib charts |
| 09 | CDN edge cache — build a cache layer on top of RustFS |
| 10 | Cluster visualizer — interactive ipywidgets dashboard |
